# Lab 2: Path finding (BFS, DFS and A*)
Environment (Anaconda): `conda activate ailab`, then `jupyter notebook`. Run cells top to bottom with **Shift+Enter**.

**Problem:** a robot moves on a grid map (4 directions, cost 1 per move) from START to GOAL. We compare how many cells each algorithm opens and how long the path is.

## Cell 1: Setup and random map

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
from collections import deque

ROWS, COLS = 10, 15          # map size
WALL_PROB = 0.25             # chance that a cell is a wall
SEED = 3                # makes the random map repeatable
START = (0, 0)               # (row, column)
GOAL = (ROWS - 1, COLS - 1)
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right

rng = np.random.default_rng(SEED)
grid = (rng.random((ROWS, COLS)) < WALL_PROB).astype(int)   # 1 = wall
grid[START] = 0
grid[GOAL] = 0

plt.imshow(grid, cmap="Greys")
plt.title("Map (black = wall)")
plt.show()

## Cell 2: BFS and DFS in one function (only `popleft` vs `pop` differs)

In [ ]:
def blind_search(grid, start, goal, mode):
    frontier = deque([start])
    came_from = {start: None}           # cell -> previous cell
    expanded = []                       # cells in the order we open them
    while frontier:
        if mode == "bfs":
            cell = frontier.popleft()   # oldest first (queue)
        else:
            cell = frontier.pop()       # newest first (stack)
        expanded.append(cell)
        if cell == goal:
            break
        for dr, dc in MOVES:
            nxt = (cell[0] + dr, cell[1] + dc)
            inside = 0 <= nxt[0] < ROWS and 0 <= nxt[1] < COLS
            if inside and grid[nxt] == 0 and nxt not in came_from:
                came_from[nxt] = cell
                frontier.append(nxt)
    if goal not in came_from:
        return None, expanded           # no path
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1], expanded

## Cell 3: A* with the Manhattan heuristic

In [ ]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(grid, start, goal):
    heap = [(manhattan(start, goal), 0, start)]   # (f, g, cell)
    came_from = {start: None}
    best_g = {start: 0}                           # cheapest known cost to reach a cell
    expanded = []
    closed = set()
    while heap:
        f, g, cell = heapq.heappop(heap)          # smallest f first
        if cell in closed:
            continue
        closed.add(cell)
        expanded.append(cell)
        if cell == goal:
            break
        for dr, dc in MOVES:
            nxt = (cell[0] + dr, cell[1] + dc)
            inside = 0 <= nxt[0] < ROWS and 0 <= nxt[1] < COLS
            if inside and grid[nxt] == 0:
                new_g = g + 1
                if nxt not in best_g or new_g < best_g[nxt]:
                    best_g[nxt] = new_g
                    came_from[nxt] = cell
                    heapq.heappush(heap, (new_g + manhattan(nxt, goal), new_g, nxt))
    if goal not in came_from:
        return None, expanded
    path = []
    cell = goal
    while cell is not None:
        path.append(cell)
        cell = came_from[cell]
    return path[::-1], expanded

## Cell 4: Run the three algorithms

In [ ]:
results = {}
for name in ["bfs", "dfs", "astar"]:
    if name == "astar":
        path, expanded = astar(grid, START, GOAL)
    else:
        path, expanded = blind_search(grid, START, GOAL, name)
    results[name] = (path, expanded)
    path_text = "no path" if path is None else f"{len(path) - 1:3d}"
    print(f"{name:6s} path length = {path_text:>8s}   expanded cells = {len(expanded):3d}")


## Cell 5: Draw expanded cells and paths

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, name in zip(axes, ["bfs", "dfs", "astar"]):
    path, expanded = results[name]
    picture = grid.astype(float)               # 0 free, 1 wall
    for cell in expanded:
        picture[cell] = 0.4                    # expanded cells
    if path is not None:
        for cell in path:
            picture[cell] = 0.7                # final path
        path_text = str(len(path) - 1)
    else:
        path_text = "no path"
    ax.imshow(picture, cmap="viridis")
    ax.set_title(f"{name}: path {path_text}, expanded {len(expanded)}")
    ax.axis("off")
plt.show()


## Cell 6: 100 random maps

In [ ]:
rows = []
for seed in range(100):
    rng = np.random.default_rng(seed)
    g = (rng.random((ROWS, COLS)) < WALL_PROB).astype(int)
    g[START] = 0
    g[GOAL] = 0
    for name in ["bfs", "dfs", "astar"]:
        if name == "astar":
            path, expanded = astar(g, START, GOAL)
        else:
            path, expanded = blind_search(g, START, GOAL, name)
        if path is not None:                      # skip maps with no path
            rows.append({"algo": name, "path": len(path) - 1, "expanded": len(expanded)})

df = pd.DataFrame(rows)
summary = df.groupby("algo")[["path", "expanded"]].mean().round(1)
print(summary)
print("solvable maps:", len(df) // 3)

## Cell 7: Chart of the averages

In [ ]:
summary.plot(kind="bar", subplots=True, layout=(1, 2), figsize=(8, 3), legend=False)
plt.show()

# Part 2: Motion output
Run Part 1 first. This cell animates the order in which each algorithm opens cells.

## Cell 8: Animation (blue = expanded, orange = final path)

In [ ]:
from matplotlib.animation import FuncAnimation
from matplotlib.colors import ListedColormap
from IPython.display import HTML

colors = ListedColormap(["white", "#1f2937", "#93c5fd", "#f59e0b"])   # free, wall, expanded, path
names = ["bfs", "dfs", "astar"]
fig, axes = plt.subplots(1, 3, figsize=(9, 3), dpi=60)

def draw(t):
    for ax, name in zip(axes, names):
        path, expanded = results[name]
        picture = grid.copy()                      # 0 free, 1 wall
        for cell in expanded[:t]:
            picture[cell] = 2
        if path is not None and t >= len(expanded):
            for cell in path:
                picture[cell] = 3
        ax.clear()
        ax.imshow(picture, cmap=colors, vmin=0, vmax=3)
        path_text = f"path {len(path) - 1}" if path is not None else "no path"
        ax.set_title(f"{name}: {min(t, len(expanded))} expanded, {path_text}", fontsize=9)
        ax.axis("off")

frames = max(len(results[n][1]) for n in names) + 10
anim = FuncAnimation(fig, draw, frames=frames, interval=80)
plt.close(fig)
HTML(anim.to_jshtml())


## Analysis questions
1. Why does A* open fewer cells?
2. Why are BFS and A* paths equally short?
3. Why is the DFS path longer?
4. What happens if WALL_PROB is 0.45?

## Challenges
More walls, different cell costs, diagonal moves, Euclidean heuristic.


# Challenge: More walls

در این بخش، چالش «موانع بیشتر» از اسلاید استاد پیاده‌سازی می‌شود. احتمال مانع بودن خانه‌ها از ۰٫۱۰ تا ۰٫۵۰ تغییر می‌کند. برای هر مقدار، ۱۰۰ نقشه تولید می‌کنیم و نسبت نقشه‌هایی را که هیچ مسیری بین مبدأ و مقصد ندارند، محاسبه می‌کنیم.

برای مقایسه منصفانه‌تر، در هر seed یک ماتریس تصادفی پایه می‌سازیم و همان ماتریس را با احتمال‌های مختلف آستانه‌گذاری می‌کنیم؛ بنابراین با افزایش احتمال مانع، تعداد موانع همان نقشه بیشتر می‌شود.


In [ ]:
wall_probabilities = np.round(np.arange(0.10, 0.51, 0.05), 2)
N_MAPS = 100
no_path_records = []
performance_records = []

for wall_prob in wall_probabilities:
    no_path_count = 0
    for seed in range(N_MAPS):
        rng = np.random.default_rng(seed)
        random_values = rng.random((ROWS, COLS))
        test_grid = (random_values < wall_prob).astype(int)
        test_grid[START] = 0
        test_grid[GOAL] = 0

        # BFS is complete for this finite, four-direction, unit-cost grid.
        bfs_path, bfs_expanded = blind_search(test_grid, START, GOAL, 'bfs')
        if bfs_path is None:
            no_path_count += 1
            continue

        dfs_path, dfs_expanded = blind_search(test_grid, START, GOAL, 'dfs')
        astar_path, astar_expanded = astar(test_grid, START, GOAL)

        performance_records.extend([
            {'wall_probability': wall_prob, 'seed': seed, 'algorithm': 'BFS',
             'path_length': len(bfs_path) - 1, 'expanded_cells': len(bfs_expanded)},
            {'wall_probability': wall_prob, 'seed': seed, 'algorithm': 'DFS',
             'path_length': len(dfs_path) - 1, 'expanded_cells': len(dfs_expanded)},
            {'wall_probability': wall_prob, 'seed': seed, 'algorithm': 'A*',
             'path_length': len(astar_path) - 1, 'expanded_cells': len(astar_expanded)}
        ])

    no_path_records.append({
        'wall_probability': wall_prob,
        'maps_without_path': no_path_count,
        'maps_with_path': N_MAPS - no_path_count,
        'no_path_ratio': no_path_count / N_MAPS
    })

no_path_df = pd.DataFrame(no_path_records)
performance_df = pd.DataFrame(performance_records)
display(no_path_df)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(no_path_df['wall_probability'], no_path_df['no_path_ratio'] * 100, marker='o')
ax.set_xlabel('Wall probability')
ax.set_ylabel('Maps without a path (%)')
ax.set_title('Effect of obstacle probability on solvability')
ax.set_xticks(wall_probabilities)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
plt.show()

if not performance_df.empty:
    performance_summary = performance_df.groupby(
        ['wall_probability', 'algorithm']
    )[['path_length', 'expanded_cells']].mean().round(2)
    display(performance_summary)


## نکات تحلیل چالش

- با افزایش احتمال مانع، انتظار داریم نسبت نقشه‌های بدون مسیر بیشتر شود؛ مقدار دقیق را از جدول و نمودار اجرای خودتان گزارش کنید.
- میانگین طول مسیر و تعداد خانه‌های بازشده فقط روی نقشه‌هایی محاسبه می‌شود که مسیر دارند؛ نقشه‌های بدون مسیر جداگانه شمرده می‌شوند.
- در این مسئله، BFS و A* با هزینه یکنواخت هر حرکت باید مسیر بهینه با طول یکسان پیدا کنند. DFS تضمین بهینگی ندارد.
- در این نسخه، بخش اصلی تمرین استاد حفظ شده و فقط مدیریت حالت «مسیر وجود ندارد» و آزمایش چالش به آن افزوده شده است.
